# AMS Sketch: Estimating Frequency Moments

## Learning Objectives

1. **Define** frequency moments $F_k$ and explain why the 2nd moment ($F_2$) matters
2. **Derive** the Alon-Matias-Szegedy estimator for $F_2$
3. **Implement** AMS sketch with median-of-means improvement
4. **Apply** AMS to self-join size estimation

## Frequency Moments

Given a stream over a universe $[m]$, let $f_i$ = frequency of item $i$.

The **$k$-th frequency moment** is:
$$F_k = \sum_{i=1}^{m} f_i^k$$

| Moment | Meaning |
|--------|---------|
| $F_0 = $ number of distinct items | cardinality |
| $F_1 = \sum f_i = n$ | stream length |
| $F_2 = \sum f_i^2$ | **surprise number** / self-join size |
| $F_{\infty} = \max_i f_i$ | most frequent item frequency |

**$F_2$** measures how concentrated the distribution is:
- Uniform: $F_2 = n^2/m$ (small)
- One item dominates: $F_2 \approx n^2$ (large)

## The AMS Estimator

**Random variable trick:**

Let $\xi_i \in \{+1, -1\}$ with equal probability, independently for each item $i$.

Define $Z = \sum_{i=1}^{m} \xi_i \cdot f_i$ (accumulated as items arrive).

Then:
$$\mathbb{E}[Z^2] = \sum_i f_i^2 + \sum_{i \neq j} \xi_i \xi_j f_i f_j = F_2 + 0 = F_2$$

**Key:** $Z$ can be maintained in $O(1)$ — when item $i$ arrives, add $\xi_i$ to $Z$.

**Variance:** high. Fix with **median of means**:
- $t$ independent estimates grouped into $s$ groups of $t/s$
- Return median of group means → $(1\pm\epsilon)$ approximation with high probability

In [ ]:
import math, random

def fourwise_hash(item, a, b, c, d, p=2**31-1):
    """4-wise independent hash → {+1, -1}."""
    h = (a * item**3 + b * item**2 + c * item + d) % p
    return 1 if h % 2 == 0 else -1

class AMSSketch:
    def __init__(self, n_rows=5, n_cols=3, seed=0):
        # n_rows: number of independent estimators per group
        # n_cols: number of groups (take median)
        self.R = n_rows; self.S = n_cols
        rng = random.Random(seed)
        p = 2**31-1
        # Random hash parameters for each (row, col)
        self.params = [[(rng.randint(1,p), rng.randint(1,p),
                          rng.randint(1,p), rng.randint(1,p)) for _ in range(n_rows)]
                       for _ in range(n_cols)]
        self.Z = [[0.0]*n_rows for _ in range(n_cols)]

    def add(self, item, count=1):
        for s in range(self.S):
            for r in range(self.R):
                a,b,c,d = self.params[s][r]
                xi = fourwise_hash(item, a, b, c, d)
                self.Z[s][r] += xi * count

    def estimate(self):
        # Mean of each group, then median of means
        group_estimates = [sum(z**2 for z in self.Z[s])/self.R for s in range(self.S)]
        group_estimates.sort()
        return group_estimates[self.S//2]

# Test
rng = random.Random(42)
# Stream with known F2
items = [1]*100 + [2]*200 + [3]*50 + [4]*150
# True F2
from collections import Counter
freq = Counter(items)
f2_true = sum(v**2 for v in freq.values())
print(f"True F2: {f2_true}")

sketch = AMSSketch(n_rows=5, n_cols=5)
rng.shuffle(items)
for x in items: sketch.add(x)
print(f"AMS estimate: {sketch.estimate():.0f}")
print(f"Relative error: {abs(sketch.estimate()-f2_true)/f2_true:.3f}")

In [ ]:
# Accuracy vs sketch size
print("(n_rows, n_cols) | Avg rel error (15 trials)")
n_stream = 500
items_test = [rng.randint(1,20) for _ in range(n_stream)]
freq_test = Counter(items_test)
f2_test = sum(v**2 for v in freq_test.values())

for n_rows, n_cols in [(1,1),(3,3),(5,5),(7,7),(9,9)]:
    errors = []
    for trial in range(15):
        sk = AMSSketch(n_rows=n_rows, n_cols=n_cols, seed=trial)
        for x in items_test: sk.add(x)
        errors.append(abs(sk.estimate()-f2_test)/f2_test)
    avg = sum(errors)/len(errors)
    space = n_rows * n_cols
    bar = '#'*int(avg*100)
    print(f"  ({n_rows},{n_cols}) space={space:>3}: avg_err={avg:.3f}  {bar}")

## Self-Join Size Application

$F_2 = \sum_i f_i^2$ equals the size of the **self-join** of the relation $R$:
$$|R \bowtie R| = \sum_{a \in \text{domain}} |\sigma_{A=a}(R)|^2 = \sum_i f_i^2 = F_2$$

This is used in **query optimization**: estimating join sizes without materializing the join.

AMS is one of the original **sketching** algorithms — it introduced the idea of using random projections to summarize data streams. The technique generalizes to arbitrary inner products and is the basis for feature hashing in machine learning.